# Reverse Logistics — Component Inspection Agent (Embedded Gradio App)

This notebook implements the **Cleaning & Inspection** step of a remanufacturing reverse-logistics
flow (Disassembly \u2192 Cleaning & Inspection \u2192 Salvaging & Machining \u2192 Assembly \u2192 Testing):
users upload one or more photos of disassembled components, and the system detects defects, classifies
each image, and applies a Pass/Reject decision \u2014 the same goal as a real-world quality-inspection
station, just powered by a vision LLM instead of a custom-trained detector.

**Categories:** `Crack`, `Corrosion`, `Wear`, `Good Area`

**Decision rule (deterministic, not left to the LLM):**
- `Crack` or `Corrosion` detected anywhere on the part \u2192 **TIDAK LULUS** (Reject / Repair)
- Only `Wear` detected \u2192 **LULUS** (Pass \u2192 Lanjut Machining)
- Only `Good Area` detected \u2192 **LULUS** (Pass \u2192 Lanjut Assembly)

**Why a vision LLM instead of a trained YOLOv8 model:** there is no labeled component image dataset
or trained model bundled in this project. Rather than block on collecting one, this notebook uses
Gemini's multimodal model to classify each uploaded image zero-shot, then runs the pass/reject logic
through a small deterministic rule function (`apply_decision_rule` in `vision_tools.py`) so the business
rule stays auditable. To move to production with a real trained detector, only `classify_component_image`
in `vision_tools.py` needs to change \u2014 every agent, the graph, and the UI stay the same.

Fully self-contained: no `git clone`, no dependency on a GitHub repo being reachable. Every module's
source code is embedded directly below and written to disk when you run the cells.


In [ ]:
# 1. Create a working directory
import os
os.makedirs("/content/reverse_logistics_agent", exist_ok=True)
os.makedirs("/content/reverse_logistics_agent/agents", exist_ok=True)
os.makedirs("/content/reverse_logistics_agent/data", exist_ok=True)
os.makedirs("/content/reverse_logistics_agent/data/uploads", exist_ok=True)
%cd /content/reverse_logistics_agent


In [ ]:
%%writefile requirements.txt
# Core LangChain ecosystem
langchain>=0.3.0,<0.4.0
langchain-core>=0.3.0,<0.4.0
langchain-community>=0.3.0,<0.4.0
langchain-openai>=0.2.0,<0.3.0
langgraph>=0.2.0,<0.3.0
langsmith>=0.1.100,<0.2.0

# Gemini (via OpenAI-compatible endpoint, free tier; multimodal for vision + chat)
openai>=1.45.0,<2.0.0

# Vector store
faiss-cpu>=1.8.0,<2.0.0

# UI
gradio

# Utilities
python-dotenv>=1.0.0
pydantic>=2.9.0,<3.0.0
pydantic-settings>=2.5.0
tiktoken>=0.8.0
pandas>=2.0.0
numpy>=1.26.0,<2.0.0
Pillow>=10.0.0
SQLAlchemy>=2.0.0
grandalf


In [ ]:
# 2. Install dependencies
# pydantic<2.11 is required: newer pydantic breaks gradio's API schema introspection
!pip install -q -r requirements.txt
!pip install -q "pydantic<2.11"


## Get a free Gemini API key

1. Go to https://aistudio.google.com/apikey
2. Click "Create API key" and copy it.
3. Run the next cell **on its own** (not via "Run all") and wait for the input box to appear at the top before pasting. The cell after it does a live test call so you'll know right away if the key works.


In [ ]:
# 3. Configure environment variables
import getpass

while True:
    google_api_key = getpass.getpass("Enter your Google AI Studio API key: ").strip()
    if not google_api_key:
        print("\u274c Empty input \u2014 the key box may not have been ready. Try again.")
        continue
    break

env_content = f"""GOOGLE_API_KEY={google_api_key}
LANGSMITH_API_KEY=your_key_here
LANGCHAIN_PROJECT=reverse-logistics-inspection
LANGCHAIN_TRACING_V2=false
LANGSMITH_TRACING=False
LANGSMITH_ENDPOINT=https://api.smith.langchain.com/
LANGSMITH_PROJECT=reverse-logistics-inspection
"""

with open(".env", "w") as f:
    f.write(env_content)

print(f"\u2713 .env saved (key length: {len(google_api_key)}). Run the next cell to verify it actually works.")


In [ ]:
# 3b. Sanity-check the key actually works before initializing anything
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv(override=True)
_client = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)
try:
    _resp = _client.chat.completions.create(
        model="gemini-2.5-flash",
        messages=[{"role": "user", "content": "Say OK"}],
    )
    print("\u2713 Key works:", _resp.choices[0].message.content)
except Exception as e:
    print("\u274c Key test failed:", e)
    print("Re-run the previous cell and paste the key again.")


## Write the application source files

Each cell below writes one module of the reverse logistics inspection agent application.


In [ ]:
%%writefile config.py
"""Configuration module for environment variables and settings."""
import os
from dotenv import load_dotenv
from pathlib import Path

# Load environment variables
load_dotenv()

# Gemini Configuration (used via Google's OpenAI-compatible endpoint, free tier)
OPENAI_API_KEY = os.getenv("GOOGLE_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("GOOGLE_API_KEY not found in environment variables")
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

# LangSmith Configuration
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
LANGSMITH_PROJECT = os.getenv("LANGSMITH_PROJECT", "reverse-logistics-inspection")
LANGSMITH_ENDPOINT = "https://api.smith.langchain.com"

# Database Configuration
DB_PATH = Path("data/reverse_logistics.db")
DB_PATH.parent.mkdir(exist_ok=True)

# Upload directory for images coming in through the Gradio UI
UPLOAD_DIR = Path("data/uploads")
UPLOAD_DIR.mkdir(exist_ok=True)

# RAG Configuration
RAG_DOCUMENTS_PATH = Path("data/qc_docs")
RAG_DOCUMENTS_PATH.mkdir(exist_ok=True)
EMBEDDING_MODEL = "gemini-embedding-001"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
TOP_K_RESULTS = 3

# Model Configuration \u2014 gemini-2.5-flash is multimodal, so the same model
# handles both image classification and the text agents below.
LLM_MODEL = "gemini-2.5-flash"
TEMPERATURE = 0.3

# Defect categories, in decision priority order (worst first), and which
# categories force a reject.
CATEGORIES = ["Crack", "Corrosion", "Wear", "Good Area"]
REJECT_CATEGORIES = {"Crack", "Corrosion"}

if __name__ == "__main__":
    # Test configuration
    print("\u2713 Configuration loaded successfully")
    print(f"  LangSmith Project: {LANGSMITH_PROJECT}")
    print(f"  Database Path: {DB_PATH}")
    print(f"  Categories: {CATEGORIES}")
    print(f"  LLM Model: {LLM_MODEL}")


In [ ]:
%%writefile db.py
"""Database module for persistent storage with multi-user support.

Tracks two things: chat conversations (for the agent UI) and per-image
inspection records (for QC traceability across upload batches).
"""
import sqlite3
import json
from datetime import datetime
from contextlib import contextmanager
from typing import List, Dict, Optional, Any
import threading
from config import DB_PATH

# Thread-local storage for database connections
_thread_local = threading.local()

class DatabaseManager:
    """Thread-safe database manager for SQLite operations."""

    def __init__(self, db_path: str = DB_PATH):
        self.db_path = str(db_path)
        self._init_db()

    def _get_connection(self):
        """Get thread-local database connection."""
        if not hasattr(_thread_local, "connection"):
            _thread_local.connection = sqlite3.connect(
                self.db_path,
                timeout=30,  # Wait up to 30s for lock
                check_same_thread=False  # We're managing threads manually
            )
            _thread_local.connection.row_factory = sqlite3.Row
        return _thread_local.connection

    @contextmanager
    def get_cursor(self):
        """Context manager for database cursors with automatic commit/rollback."""
        conn = self._get_connection()
        cursor = conn.cursor()
        try:
            yield cursor
            conn.commit()
        except Exception:
            conn.rollback()
            raise
        finally:
            cursor.close()

    def _init_db(self):
        """Initialize database tables if they don't exist."""
        with self.get_cursor() as cursor:
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS conversations (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    session_id TEXT NOT NULL,
                    user_query TEXT NOT NULL,
                    assistant_response TEXT NOT NULL,
                    agent_used TEXT,
                    timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
                    metadata TEXT
                )
            """)
            cursor.execute("""
                CREATE INDEX IF NOT EXISTS idx_session_timestamp
                ON conversations(session_id, timestamp)
            """)
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS inspections (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    batch_id TEXT NOT NULL,
                    filename TEXT NOT NULL,
                    component_type TEXT,
                    categories_found TEXT,
                    decision TEXT,
                    action TEXT,
                    driving_category TEXT,
                    detections TEXT,
                    timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
                )
            """)
            cursor.execute("""
                CREATE INDEX IF NOT EXISTS idx_batch
                ON inspections(batch_id, timestamp)
            """)

    def save_conversation(self, session_id: str, user_query: str,
                         assistant_response: str, agent_used: str = None,
                         metadata: Dict = None):
        """Save a conversation turn to database."""
        with self.get_cursor() as cursor:
            cursor.execute("""
                INSERT INTO conversations
                (session_id, user_query, assistant_response, agent_used, metadata)
                VALUES (?, ?, ?, ?, ?)
            """, (
                session_id,
                user_query,
                assistant_response,
                agent_used,
                json.dumps(metadata) if metadata else None
            ))

    def load_session_history(self, session_id: str, limit: int = 50) -> List[Dict]:
        """Load conversation history for a session."""
        with self.get_cursor() as cursor:
            cursor.execute("""
                SELECT user_query, assistant_response, agent_used, timestamp, metadata
                FROM conversations
                WHERE session_id = ?
                ORDER BY timestamp DESC
                LIMIT ?
            """, (session_id, limit))

            rows = cursor.fetchall()
            return [
                {
                    "user_query": row["user_query"],
                    "assistant_response": row["assistant_response"],
                    "agent_used": row["agent_used"],
                    "timestamp": row["timestamp"],
                    "metadata": json.loads(row["metadata"]) if row["metadata"] else {}
                }
                for row in rows
            ]

    def get_all_sessions(self) -> List[str]:
        """Get all unique session IDs."""
        with self.get_cursor() as cursor:
            cursor.execute("SELECT DISTINCT session_id FROM conversations")
            return [row["session_id"] for row in cursor.fetchall()]

    def delete_session(self, session_id: str):
        """Delete a session and all its conversations."""
        with self.get_cursor() as cursor:
            cursor.execute("DELETE FROM conversations WHERE session_id = ?", (session_id,))

    def save_inspection(self, batch_id: str, filename: str, component_type: str,
                        categories_found: List[str], decision: str, action: str,
                        driving_category: str, detections: List[Dict[str, Any]]):
        """Save a single image's inspection result to the database."""
        with self.get_cursor() as cursor:
            cursor.execute("""
                INSERT INTO inspections
                (batch_id, filename, component_type, categories_found, decision, action, driving_category, detections)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                batch_id, filename, component_type,
                json.dumps(categories_found), decision, action, driving_category,
                json.dumps(detections)
            ))

    def get_batch_inspections(self, batch_id: str) -> List[Dict[str, Any]]:
        """Fetch all inspection records belonging to a batch, oldest first."""
        with self.get_cursor() as cursor:
            cursor.execute("""
                SELECT filename, component_type, categories_found, decision, action, driving_category, detections, timestamp
                FROM inspections
                WHERE batch_id = ?
                ORDER BY timestamp ASC
            """, (batch_id,))
            rows = cursor.fetchall()
            return [
                {
                    "filename": row["filename"],
                    "component_type": row["component_type"],
                    "categories_found": json.loads(row["categories_found"]) if row["categories_found"] else [],
                    "decision": row["decision"],
                    "action": row["action"],
                    "driving_category": row["driving_category"],
                    "detections": json.loads(row["detections"]) if row["detections"] else [],
                    "timestamp": row["timestamp"]
                }
                for row in rows
            ]

    def get_latest_batch_id(self) -> Optional[str]:
        """Return the most recently inspected batch id, if any."""
        with self.get_cursor() as cursor:
            cursor.execute("SELECT batch_id FROM inspections ORDER BY timestamp DESC LIMIT 1")
            row = cursor.fetchone()
            return row["batch_id"] if row else None


# Global database instance
db_manager = DatabaseManager()

if __name__ == "__main__":
    # Test database functionality
    print("Testing Database Module...")

    test_batch = "test_batch_123"
    db_manager.save_inspection(
        batch_id=test_batch,
        filename="engine_block_01.jpg",
        component_type="engine_block",
        categories_found=["Crack", "Corrosion", "Good Area"],
        decision="TIDAK LULUS",
        action="Reject / Repair",
        driving_category="Crack",
        detections=[{"category": "Crack", "confidence": 0.92, "description": "hairline crack near bolt hole"}]
    )

    records = db_manager.get_batch_inspections(test_batch)
    print(f"\u2713 Saved and loaded {len(records)} inspection records")
    print(f"\u2713 Latest batch id: {db_manager.get_latest_batch_id()}")


In [ ]:
%%writefile vision_tools.py
"""Vision-based defect detection tool: classifies component images with Gemini's
multimodal model and applies a deterministic, rule-based pass/reject decision.

There is no labeled image dataset or trained YOLOv8 model bundled with this project,
so detection here is zero-shot via an LLM vision call rather than a custom-trained
classifier. Swap classify_component_image's body for a real model's inference call
to go to production \u2014 every agent, the graph, and the UI stay the same.
"""
import base64
import json
import uuid
from pathlib import Path
from typing import Dict, Any, List, Optional
from openai import OpenAI
from langsmith import traceable
from config import OPENAI_API_KEY, GEMINI_BASE_URL, LLM_MODEL, CATEGORIES, REJECT_CATEGORIES
from db import db_manager

# Initialize OpenAI-compatible client (pointed at Gemini)
client = OpenAI(api_key=OPENAI_API_KEY, base_url=GEMINI_BASE_URL)

def _encode_image(image_path: str) -> str:
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def _mime_type(image_path: str) -> str:
    suffix = Path(image_path).suffix.lstrip(".").lower()
    if suffix in ("jpg", "jpeg"):
        return "image/jpeg"
    return f"image/{suffix or 'jpeg'}"

@traceable(name="classify_component_image", run_type="chain")
def classify_component_image(image_path: str) -> Dict[str, Any]:
    """Zero-shot defect detection on a single component image via Gemini vision."""
    b64 = _encode_image(image_path)
    mime = _mime_type(image_path)

    system_prompt = f"""You are a visual QC inspector for remanufactured mechanical components
    (engine blocks, crankshafts, connecting rods, and similar parts). Examine the image and
    identify every visible defect region, classifying each into exactly one of: {", ".join(CATEGORIES)}.

    Respond with JSON only, in this exact shape:
    {{"component_type": "short guess at the part name", "detections": [{{"category": "one of the categories above", "confidence": 0.0-1.0, "description": "brief description of what you see"}}]}}

    If the part looks defect-free, return a single detection with category "Good Area".
    """

    response = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=0.2,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": "Inspect this component image for defects."},
                    {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}}
                ]
            }
        ]
    )

    try:
        result = json.loads(response.choices[0].message.content)
    except Exception:
        result = {
            "component_type": "unknown",
            "detections": [{"category": "Good Area", "confidence": 0.0, "description": "Failed to parse model output"}]
        }

    result.setdefault("detections", [])
    result["usage"] = response.usage.model_dump() if response.usage else None
    return result

def apply_decision_rule(detections: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Deterministic pass/reject decision, kept separate from the LLM so the
    business rule is auditable and never depends on model phrasing."""
    categories_found = sorted({d.get("category", "Good Area") for d in detections}) or ["Good Area"]
    driving_category = next((c for c in CATEGORIES if c in categories_found), "Good Area")

    if driving_category in REJECT_CATEGORIES:
        decision = "TIDAK LULUS"
        action = "Reject / Repair"
    elif driving_category == "Wear":
        decision = "LULUS"
        action = "Lanjut Machining"
    else:
        decision = "LULUS"
        action = "Lanjut Assembly"

    return {
        "decision": decision,
        "action": action,
        "driving_category": driving_category,
        "categories_found": categories_found
    }

@traceable(name="classify_batch", run_type="chain")
def classify_batch(image_paths: List[str], batch_id: Optional[str] = None) -> Dict[str, Any]:
    """Classify a batch of uploaded images and persist each result to the DB."""
    batch_id = batch_id or str(uuid.uuid4())
    records = []

    for image_path in image_paths:
        result = classify_component_image(image_path)
        rule = apply_decision_rule(result["detections"])
        record = {
            "batch_id": batch_id,
            "filename": Path(image_path).name,
            "component_type": result.get("component_type", "unknown"),
            "detections": result["detections"],
            **rule
        }
        db_manager.save_inspection(
            batch_id=batch_id,
            filename=record["filename"],
            component_type=record["component_type"],
            categories_found=record["categories_found"],
            decision=record["decision"],
            action=record["action"],
            driving_category=record["driving_category"],
            detections=record["detections"]
        )
        records.append(record)

    return {"batch_id": batch_id, "records": records}

def get_batch_inspections(batch_id: Optional[str] = None) -> List[Dict[str, Any]]:
    """Fetch all inspection records for a batch (defaults to the most recent batch)."""
    batch_id = batch_id or db_manager.get_latest_batch_id()
    if not batch_id:
        return []
    return db_manager.get_batch_inspections(batch_id)

def get_latest_batch_id() -> Optional[str]:
    """Return the most recently inspected batch id, if any."""
    return db_manager.get_latest_batch_id()

def get_batch_summary(batch_id: Optional[str] = None) -> Dict[str, Any]:
    """Aggregate pass/reject stats and category frequency for a batch."""
    records = get_batch_inspections(batch_id)
    if not records:
        return {"batch_id": batch_id, "n_items": 0}

    n_pass = sum(1 for r in records if r["decision"] == "LULUS")
    n_reject = len(records) - n_pass
    category_counts: Dict[str, int] = {}
    for r in records:
        for c in r["categories_found"]:
            category_counts[c] = category_counts.get(c, 0) + 1

    return {
        "batch_id": batch_id or get_latest_batch_id(),
        "n_items": len(records),
        "n_pass": n_pass,
        "n_reject": n_reject,
        "category_counts": category_counts
    }

if __name__ == "__main__":
    print("Testing decision rule (no API call needed)...")
    sample_reject = [
        {"category": "Crack", "confidence": 0.92, "description": "hairline crack"},
        {"category": "Corrosion", "confidence": 0.88, "description": "surface rust"},
    ]
    print(apply_decision_rule(sample_reject))
    print(apply_decision_rule([{"category": "Wear", "confidence": 0.7}]))
    print(apply_decision_rule([{"category": "Good Area", "confidence": 0.95}]))


In [ ]:
%%writefile rag.py
"""RAG module for document retrieval and context injection."""
import os
from typing import List
from openai import OpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langsmith import traceable
from config import (
    OPENAI_API_KEY,
    GEMINI_BASE_URL,
    EMBEDDING_MODEL,
    CHUNK_SIZE,
    CHUNK_OVERLAP,
    TOP_K_RESULTS,
)

# Initialize OpenAI-compatible client (pointed at Gemini)
client = OpenAI(api_key=OPENAI_API_KEY, base_url=GEMINI_BASE_URL)
embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    api_key=OPENAI_API_KEY,
    base_url=GEMINI_BASE_URL,
    check_embedding_ctx_length=False,
)

class RAGSystem:
    """Lightweight RAG system for reverse logistics / remanufacturing QC documents."""

    def __init__(self, persist_directory: str = "data/faiss_index"):
        self.persist_directory = persist_directory
        self.vector_store = None
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP,
            separators=["\n\n", "\n", ".", "!", "?", ",", " ", ""]
        )
        self._initialize_vector_store()

    def _initialize_vector_store(self):
        """Initialize FAISS vector store from existing index or create new."""
        if os.path.exists(self.persist_directory):
            try:
                self.vector_store = FAISS.load_local(
                    self.persist_directory,
                    embeddings,
                    allow_dangerous_deserialization=True
                )
                print(f"\u2713 Loaded existing FAISS index from {self.persist_directory}")
            except Exception as e:
                print(f"! Could not load existing index: {e}")
                self.vector_store = None

        if self.vector_store is None:
            self._create_sample_documents()

    def _create_sample_documents(self):
        """Create the reverse logistics / remanufacturing QC knowledge base."""
        documents = [
            Document(
                page_content="""Remanufacturing Process Overview:
                - The flow is Disassembly -> Cleaning & Inspection -> Salvaging & Machining -> Assembly -> Testing.
                - Defect detection happens at the Cleaning & Inspection stage, before any part enters Salvaging & Machining.
                - Only components that pass inspection (LULUS) continue down the line; rejected components are pulled for repair or scrap.
                - Keeping inspection at this single gate avoids wasting machining/assembly effort on parts that will fail later anyway.""",
                metadata={"category": "process", "source": "process_overview"}
            ),
            Document(
                page_content="""Pass/Reject Decision Rule:
                - If a component has any Crack or Corrosion detected anywhere on it, the decision is TIDAK LULUS (Reject / Repair), regardless of how many Good Area regions are also present.
                - If the only defect present is Wear, the decision is LULUS (Pass), routed to Lanjut Machining.
                - If only Good Area is detected (no defects at all), the decision is LULUS (Pass), routed to Lanjut Assembly.
                - The worst category found on a component always determines its outcome \u2014 a single crack overrides ten good regions.""",
                metadata={"category": "decision_rule", "source": "qc_rules"}
            ),
            Document(
                page_content="""Repair & Rework Guidance by Category:
                - Crack: assess location and severity before deciding weld-repair vs. scrap; cracks in load-bearing or critical areas (e.g. cylinder walls, main bearing journals) should default to scrap rather than repair.
                - Corrosion: clean with abrasive blasting or chemical treatment, then re-inspect the same component before it re-enters the line; do not assume one cleaning pass is sufficient.
                - Wear: compare measured wear against the OEM tolerance spec for that part before sending to machining; wear beyond tolerance should be reclassified as reject even though the visual category is "Wear".
                - Always re-run inspection after any repair step \u2014 do not assume a repaired part automatically becomes Good Area.""",
                metadata={"category": "repair_guidance", "source": "repair_playbook"}
            ),
            Document(
                page_content="""Traceability & QC Reporting:
                - Every inspected image should be tied to a batch id and a filename so results can be audited later.
                - A batch QC report should include: total items inspected, pass count, reject count, and a breakdown of how often each defect category appeared.
                - Use traceability records to spot a part type or supplier batch with an unusually high reject rate, not just to log individual outcomes.""",
                metadata={"category": "traceability", "source": "qc_reporting"}
            ),
            Document(
                page_content="""Benefits of Automated Visual Inspection:
                - Faster and more consistent than manual visual inspection, especially across large batches.
                - Reduces human error and inspector fatigue effects on judgment calls.
                - Ensures only quality-checked parts proceed to machining and assembly, cutting downstream rework and downtime cost.
                - Produces a structured, queryable record for every component inspected, which manual inspection typically does not.""",
                metadata={"category": "benefits", "source": "benefits_summary"}
            ),
        ]
        self.add_documents(documents)
        print("\u2713 Created sample documents and FAISS index")

    @traceable(name="rag_add_documents", run_type="chain")
    def add_documents(self, documents: List[Document]):
        """Add documents to the vector store."""
        chunks = self.text_splitter.split_documents(documents)
        if self.vector_store is None:
            self.vector_store = FAISS.from_documents(chunks, embeddings)
        else:
            self.vector_store.add_documents(chunks)
        os.makedirs(os.path.dirname(self.persist_directory) or ".", exist_ok=True)
        self.vector_store.save_local(self.persist_directory)

    @traceable(name="rag_retrieve", run_type="retriever")
    def retrieve_context(self, query: str, k: int = TOP_K_RESULTS) -> List[Document]:
        """Retrieve relevant documents for a query."""
        if self.vector_store is None:
            return []
        return self.vector_store.similarity_search(query, k=k)

    @traceable(name="rag_get_context", run_type="chain")
    def get_context_string(self, query: str) -> str:
        """Get context as a formatted string for prompt injection."""
        docs = self.retrieve_context(query)
        if not docs:
            return "No relevant documents found."
        context_parts = []
        for i, doc in enumerate(docs, 1):
            source = doc.metadata.get("source", "Unknown")
            category = doc.metadata.get("category", "General")
            context_parts.append(f"[Document {i} - {category} ({source})]:\n{doc.page_content}\n")
        return "\n".join(context_parts)


# Global RAG instance
rag_system = RAGSystem()

if __name__ == "__main__":
    print("Testing RAG Module...")
    test_queries = [
        "What happens when both Crack and Good Area are detected on the same part?",
        "How should corroded components be repaired?",
        "What should a batch QC report include?",
    ]
    for query in test_queries:
        print(f"\nQuery: {query}")
        print("-" * 50)
        context = rag_system.get_context_string(query)
        print(f"Retrieved Context:\n{context}")


In [ ]:
%%writefile router.py
"""Router module for classifying user intent in reverse logistics inspection queries."""
import json
from typing import Dict, Any
from openai import OpenAI
from langsmith import traceable
from config import OPENAI_API_KEY, GEMINI_BASE_URL, LLM_MODEL

# Initialize OpenAI-compatible client (pointed at Gemini)
client = OpenAI(api_key=OPENAI_API_KEY, base_url=GEMINI_BASE_URL)

@traceable(name="router_classification", run_type="chain")
def classify_intent(query: str) -> Dict[str, Any]:
    """
    Classify user query into one of the reverse logistics inspection categories.

    Args:
        query: User's question

    Returns:
        Dictionary with classification result and confidence
    """
    system_prompt = """You are an intent classifier for a Reverse Logistics Component Inspection System.
    Classify the user's query into one of these categories:

    1. inspection_explain - Questions about why a specific image or component was classified/flagged a certain way
    2. qc_report - Questions asking for aggregate stats across the current batch (pass/reject counts, category breakdown)
    3. repair_guidance - Questions about how to repair, rework, or handle a flagged component
    4. general - General questions not specific to the above categories

    Respond with JSON format: {"category": "category_name", "confidence": 0.0-1.0, "reasoning": "brief explanation"}
    """

    response = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=0.2,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query}
        ]
    )

    try:
        result = json.loads(response.choices[0].message.content)
    except Exception:
        result = {
            "category": "general",
            "confidence": 0.5,
            "reasoning": "Failed to parse response"
        }

    result["usage"] = response.usage.model_dump() if response.usage else None
    return result

@traceable(name="router_decision", run_type="chain")
def route_query(query: str) -> str:
    """
    Route query to appropriate agent based on intent.

    Args:
        query: User's question

    Returns:
        Agent name to route to
    """
    classification = classify_intent(query)
    return classification.get("category", "general")

if __name__ == "__main__":
    print("Testing Router Module...")

    test_queries = [
        "Why was engine_block_01.jpg rejected?",
        "How many parts passed in this batch?",
        "How should I repair a part with corrosion?",
        "What does this system do?"
    ]

    for query in test_queries:
        print(f"\nQuery: {query}")
        print("-" * 50)
        classification = classify_intent(query)
        print(f"Classification: {classification}")
        print(f"Routed to: {route_query(query)}")


In [ ]:
%%writefile agents/__init__.py



In [ ]:
%%writefile agents/inspection_agent.py
"""Inspection Explanation Agent for reverse logistics QC."""
from typing import Dict, Any, Optional
from openai import OpenAI
from langsmith import traceable
from config import OPENAI_API_KEY, GEMINI_BASE_URL, LLM_MODEL, TEMPERATURE
from vision_tools import get_batch_inspections

# Initialize OpenAI-compatible client (pointed at Gemini)
client = OpenAI(api_key=OPENAI_API_KEY, base_url=GEMINI_BASE_URL)

@traceable(name="inspection_agent", run_type="chain")
def inspection_agent(query: str, batch_id: Optional[str] = None, context: str = None) -> Dict[str, Any]:
    """
    Inspection Agent - Explains why specific images in the current batch were classified/flagged as they were.

    Args:
        query: User query about a specific image or set of images
        batch_id: Batch to pull real detection results from (defaults to the latest batch)
        context: Retrieved RAG context

    Returns:
        Dictionary with response and metadata
    """
    records = get_batch_inspections(batch_id)

    system_prompt = """You are a Remanufacturing QC Inspection Expert. Your role is to:
    - Explain why a specific component image was classified Crack/Corrosion/Wear/Good Area
    - Quote the actual detections and decision provided in the tool data, never invent findings
    - Reference the decision rule (Crack or Corrosion anywhere -> Reject/Repair; only Wear or only Good Area -> Pass) when explaining outcomes
    - If the user asks about a filename not present in the tool data, say so rather than guessing

    Use the provided tool data (real per-image detection results from this batch) and context documents to ground your answer."""

    user_prompt = f"""Tool data (this batch's detection results):
    {records}

    Context from QC guidelines:
    {context if context else "No specific context provided."}

    User Question: {query}

    Please explain the inspection outcome(s)."""

    response = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=TEMPERATURE,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    return {
        "agent": "inspection_explain",
        "response": response.choices[0].message.content,
        "usage": response.usage.model_dump() if response.usage else None
    }

if __name__ == "__main__":
    print("Testing Inspection Agent...")

    test_query = "Why might a component get flagged TIDAK LULUS?"
    result = inspection_agent(test_query, context="Crack or Corrosion anywhere forces a reject.")

    print(f"\nQuery: {test_query}")
    print(f"\nResponse: {result['response']}")
    print(f"\nToken Usage: {result['usage']}")


In [ ]:
%%writefile agents/qc_report_agent.py
"""QC Report Agent for reverse logistics inspection batches."""
from typing import Dict, Any, Optional
from openai import OpenAI
from langsmith import traceable
from config import OPENAI_API_KEY, GEMINI_BASE_URL, LLM_MODEL, TEMPERATURE
from vision_tools import get_batch_summary

# Initialize OpenAI-compatible client (pointed at Gemini)
client = OpenAI(api_key=OPENAI_API_KEY, base_url=GEMINI_BASE_URL)

@traceable(name="qc_report_agent", run_type="chain")
def qc_report_agent(query: str, batch_id: Optional[str] = None, context: str = None) -> Dict[str, Any]:
    """
    QC Report Agent - Summarizes pass/reject stats and defect-category breakdown for a batch.

    Args:
        query: User query asking for batch-level stats
        batch_id: Batch to summarize (defaults to the latest batch)
        context: Retrieved RAG context

    Returns:
        Dictionary with response and metadata
    """
    summary = get_batch_summary(batch_id)

    system_prompt = """You are a Remanufacturing QC Reporting Analyst. Your role is to:
    - Summarize pass/reject counts and defect-category frequency for the current batch using the real aggregates in the tool data
    - Call out which defect category drove the most rejects, if any
    - Be precise with numbers from the tool data, never invent figures
    - If the batch is empty, say no inspections have been run yet rather than fabricating numbers

    Use the provided tool data (real aggregated results) and context documents to ground your answer."""

    user_prompt = f"""Tool data (real batch aggregates):
    {summary}

    Context from QC guidelines:
    {context if context else "No specific context provided."}

    User Question: {query}

    Please provide a grounded QC summary."""

    response = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=TEMPERATURE,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    return {
        "agent": "qc_report",
        "response": response.choices[0].message.content,
        "usage": response.usage.model_dump() if response.usage else None
    }

if __name__ == "__main__":
    print("Testing QC Report Agent...")

    test_query = "Give me a summary of this batch's results."
    result = qc_report_agent(test_query, context="Include pass/reject counts and the top defect category.")

    print(f"\nQuery: {test_query}")
    print(f"\nResponse: {result['response']}")
    print(f"\nToken Usage: {result['usage']}")


In [ ]:
%%writefile agents/repair_guidance_agent.py
"""Repair Guidance Agent for reverse logistics QC."""
from typing import Dict, Any, Optional
from openai import OpenAI
from langsmith import traceable
from config import OPENAI_API_KEY, GEMINI_BASE_URL, LLM_MODEL, TEMPERATURE
from vision_tools import get_batch_inspections

# Initialize OpenAI-compatible client (pointed at Gemini)
client = OpenAI(api_key=OPENAI_API_KEY, base_url=GEMINI_BASE_URL)

@traceable(name="repair_guidance_agent", run_type="chain")
def repair_guidance_agent(query: str, batch_id: Optional[str] = None, context: str = None) -> Dict[str, Any]:
    """
    Repair Guidance Agent - Recommends repair/rework steps for rejected components in the current batch.

    Args:
        query: User query about how to handle flagged components
        batch_id: Batch to pull rejected items from (defaults to the latest batch)
        context: Retrieved RAG context

    Returns:
        Dictionary with response and metadata
    """
    records = get_batch_inspections(batch_id)
    rejected = [r for r in records if r.get("decision") == "TIDAK LULUS"]

    system_prompt = """You are a Remanufacturing Repair & Rework Expert. Your role is to:
    - Recommend concrete repair/rework actions for each rejected component, tailored to its actual flagged categories
    - Distinguish Crack handling (assess severity/location, weld-repair vs. scrap) from Corrosion handling (cleaning/treatment then re-inspect)
    - Note that any repaired part must be re-inspected before being marked Good Area
    - If no components were rejected in this batch, say so rather than inventing repair steps

    Use the provided tool data (real rejected items from this batch) and context documents to ground your recommendations."""

    user_prompt = f"""Tool data (rejected items in this batch):
    {rejected}

    Context from QC guidelines:
    {context if context else "No specific context provided."}

    User Question: {query}

    Please provide repair/rework guidance."""

    response = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=TEMPERATURE,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    return {
        "agent": "repair_guidance",
        "response": response.choices[0].message.content,
        "usage": response.usage.model_dump() if response.usage else None
    }

if __name__ == "__main__":
    print("Testing Repair Guidance Agent...")

    test_query = "How should we handle the rejected parts in this batch?"
    result = repair_guidance_agent(test_query, context="Cracks in critical areas should default to scrap.")

    print(f"\nQuery: {test_query}")
    print(f"\nResponse: {result['response']}")
    print(f"\nToken Usage: {result['usage']}")


In [ ]:
%%writefile graph.py
"""LangGraph workflow for the Reverse Logistics Inspection Assistant."""
from typing import Dict, Any, Literal
from langgraph.graph import StateGraph, END
from typing_extensions import TypedDict
from langsmith import traceable

from agents.inspection_agent import inspection_agent
from agents.qc_report_agent import qc_report_agent
from agents.repair_guidance_agent import repair_guidance_agent
from router import classify_intent
from rag import rag_system
from db import db_manager
from openai import OpenAI
from config import OPENAI_API_KEY, GEMINI_BASE_URL, LLM_MODEL

# Initialize OpenAI-compatible client (pointed at Gemini)
client = OpenAI(api_key=OPENAI_API_KEY, base_url=GEMINI_BASE_URL)

# Define state schema with conversation context
class AgentState(TypedDict):
    """State for the LangGraph agent workflow."""
    query: str
    session_id: str
    batch_id: str
    conversation_context: str
    intent: str
    intent_confidence: float
    intent_reasoning: str
    rag_context: str
    agent_response: str
    agent_used: str
    usage: Dict
    error: str

@traceable(name="inject_context", run_type="chain")
def inject_context_node(state: AgentState) -> AgentState:
    """Node to inject RAG context based on intent."""
    try:
        query = state["query"]
        context = state.get("conversation_context", "")

        enhanced_query = query
        if context:
            context_lines = context.split("\n")
            last_user_msg = None
            for line in reversed(context_lines):
                if line.startswith("User:"):
                    last_user_msg = line.replace("User:", "").strip()
                    break
            if last_user_msg:
                enhanced_query = f"Previous question: {last_user_msg}\nCurrent question: {query}"

        state["rag_context"] = rag_system.get_context_string(enhanced_query)
    except Exception as e:
        state["rag_context"] = ""
        state["error"] = f"RAG error: {str(e)}"

    return state

@traceable(name="router_node", run_type="chain")
def router_node(state: AgentState) -> AgentState:
    """Node to classify intent with conversation context."""
    try:
        query = state["query"]
        context = state.get("conversation_context", "")

        enhanced_query = query
        if context:
            context_lines = context.split("\n")
            recent_exchanges = context_lines[-4:] if len(context_lines) > 4 else context_lines
            enhanced_query = f"""Previous conversation:
{chr(10).join(recent_exchanges)}

Current question: {query}"""

        classification = classify_intent(enhanced_query)
        state["intent"] = classification.get("category", "general")
        state["intent_confidence"] = classification.get("confidence", 0.0)
        state["intent_reasoning"] = classification.get("reasoning", "")
        state["usage"] = classification.get("usage")
    except Exception as e:
        state["intent"] = "general"
        state["error"] = f"Router error: {str(e)}"

    return state

def _build_enhanced_context(state: AgentState) -> str:
    rag_context = state["rag_context"]
    conversation_context = state.get("conversation_context", "")
    if conversation_context:
        return f"""Previous conversation context:
{conversation_context}

Relevant documentation:
{rag_context}"""
    return rag_context

@traceable(name="inspection_agent_node", run_type="chain")
def inspection_agent_node(state: AgentState) -> AgentState:
    """Node for the inspection explanation agent."""
    result = inspection_agent(state["query"], state.get("batch_id"), _build_enhanced_context(state))
    state["agent_response"] = result["response"]
    state["agent_used"] = result["agent"]
    state["usage"] = result["usage"]
    return state

@traceable(name="qc_report_agent_node", run_type="chain")
def qc_report_agent_node(state: AgentState) -> AgentState:
    """Node for the QC report agent."""
    result = qc_report_agent(state["query"], state.get("batch_id"), _build_enhanced_context(state))
    state["agent_response"] = result["response"]
    state["agent_used"] = result["agent"]
    state["usage"] = result["usage"]
    return state

@traceable(name="repair_guidance_agent_node", run_type="chain")
def repair_guidance_agent_node(state: AgentState) -> AgentState:
    """Node for the repair guidance agent."""
    result = repair_guidance_agent(state["query"], state.get("batch_id"), _build_enhanced_context(state))
    state["agent_response"] = result["response"]
    state["agent_used"] = result["agent"]
    state["usage"] = result["usage"]
    return state

@traceable(name="general_agent_node", run_type="chain")
def general_agent_node(state: AgentState) -> AgentState:
    """Node for handling general queries with conversation context."""
    query = state["query"]
    context = state.get("conversation_context", "")

    messages = [
        {"role": "system", "content": "You are a helpful assistant for a reverse logistics component inspection system. Provide general information and guide users to specific agents for detailed queries."}
    ]
    if context:
        messages.append({"role": "system", "content": f"Previous conversation:\n{context}"})
    messages.append({"role": "user", "content": query})

    response = client.chat.completions.create(model=LLM_MODEL, messages=messages)

    state["agent_response"] = response.choices[0].message.content
    state["agent_used"] = "general"
    state["usage"] = response.usage.model_dump() if response.usage else None
    return state

@traceable(name="save_to_db", run_type="chain")
def save_to_db_node(state: AgentState) -> AgentState:
    """Node to save conversation to database."""
    try:
        db_manager.save_conversation(
            session_id=state["session_id"],
            user_query=state["query"],
            assistant_response=state["agent_response"],
            agent_used=state["agent_used"],
            metadata={
                "intent": state["intent"],
                "confidence": state["intent_confidence"],
                "batch_id": state.get("batch_id"),
                "conversation_context": state.get("conversation_context", ""),
                "usage": state["usage"]
            }
        )
    except Exception as e:
        state["error"] = f"Database error: {str(e)}"

    return state

def should_continue(state: AgentState) -> Literal["inspection", "qc_report", "repair", "general", END]:
    """Conditional edge to route to appropriate agent."""
    if state.get("error"):
        return END

    intent = state.get("intent", "general")

    if intent == "inspection_explain":
        return "inspection"
    elif intent == "qc_report":
        return "qc_report"
    elif intent == "repair_guidance":
        return "repair"
    else:
        return "general"

def build_inspection_graph():
    """Build and compile the LangGraph workflow."""
    workflow = StateGraph(AgentState)

    workflow.add_node("router", router_node)
    workflow.add_node("inject_context", inject_context_node)
    workflow.add_node("inspection", inspection_agent_node)
    workflow.add_node("qc_report", qc_report_agent_node)
    workflow.add_node("repair", repair_guidance_agent_node)
    workflow.add_node("general", general_agent_node)
    workflow.add_node("save_db", save_to_db_node)

    workflow.set_entry_point("router")
    workflow.add_edge("router", "inject_context")

    workflow.add_conditional_edges(
        "inject_context",
        should_continue,
        {
            "inspection": "inspection",
            "qc_report": "qc_report",
            "repair": "repair",
            "general": "general",
            END: END
        }
    )

    workflow.add_edge("inspection", "save_db")
    workflow.add_edge("qc_report", "save_db")
    workflow.add_edge("repair", "save_db")
    workflow.add_edge("general", "save_db")
    workflow.add_edge("save_db", END)

    return workflow.compile()

# Create global graph instance
inspection_graph = build_inspection_graph()

if __name__ == "__main__":

    ascii_data = inspection_graph.get_graph().draw_ascii()
    print(ascii_data)

    test_queries = [
        ("test_session_1", "What does TIDAK LULUS mean?"),
        ("test_session_1", "How many items passed in this batch?"),
        ("test_session_2", "How should we repair a cracked connecting rod?"),
        ("test_session_3", "What does this system do?")
    ]

    for session_id, query in test_queries:
        print(f"\nProcessing: {query}")
        print("-" * 50)

        initial_state = {
            "query": query,
            "session_id": session_id,
            "batch_id": None,
            "conversation_context": "",
            "intent": "",
            "intent_confidence": 0.0,
            "intent_reasoning": "",
            "rag_context": "",
            "agent_response": "",
            "agent_used": "",
            "usage": {},
            "error": ""
        }

        result = inspection_graph.invoke(initial_state)

        print(f"Intent: {result['intent']} (confidence: {result['intent_confidence']:.2f})")
        print(f"Agent Used: {result['agent_used']}")
        print(f"Response: {result['agent_response'][:100]}...")
        print("\u2713 Graph execution complete")


In [ ]:
%%writefile gradio_ui.py
"""
Reverse Logistics Inspection Assistant - UI
Run with: python gradio_ui.py
"""
import uuid
import pandas as pd
import gradio as gr
from vision_tools import classify_batch
from graph import inspection_graph
from db import db_manager

# Store session per user
sessions = {}

def run_inspection(files, batch_id_state):
    """Classify every uploaded image as one batch and show the results."""
    if not files:
        return None, pd.DataFrame(), batch_id_state, "Upload at least one image first."

    image_paths = [f.name if hasattr(f, "name") else f for f in files]
    batch_id = str(uuid.uuid4())
    result = classify_batch(image_paths, batch_id=batch_id)

    gallery_items = []
    rows = []
    for record in result["records"]:
        caption = f"{record['decision']} \u2014 {record['driving_category']} ({record['action']})"
        gallery_items.append((next(p for p in image_paths if p.endswith(record["filename"])), caption))
        rows.append({
            "filename": record["filename"],
            "component_type": record["component_type"],
            "categories_found": ", ".join(record["categories_found"]),
            "decision": record["decision"],
            "action": record["action"],
        })

    df = pd.DataFrame(rows)
    n_pass = sum(1 for r in result["records"] if r["decision"] == "LULUS")
    status = f"Batch {batch_id[:8]}: {len(result['records'])} items inspected, {n_pass} passed, {len(result['records']) - n_pass} rejected."

    return gallery_items, df, batch_id, status

def process_query(query, history, session_id, batch_id):
    """Process a single chat query against the current batch and return response."""

    if not session_id:
        session_id = str(uuid.uuid4())
        sessions[session_id] = []

    conversation_context = ""
    if history:
        context_parts = []
        for msg in history:
            if isinstance(msg, dict) and "role" in msg and "content" in msg:
                role = "User" if msg["role"] == "user" else "Assistant"
                context_parts.append(f"{role}: {msg['content']}")
        conversation_context = "\n".join(context_parts[-6:])

    state = {
        "query": query,
        "session_id": session_id,
        "batch_id": batch_id,
        "conversation_context": conversation_context,
        "intent": "",
        "intent_confidence": 0.0,
        "intent_reasoning": "",
        "rag_context": "",
        "agent_response": "",
        "agent_used": "",
        "usage": {},
        "error": ""
    }

    try:
        result = inspection_graph.invoke(state)

        if result.get("error"):
            response = f"\u274c Error: {result['error']}"
        else:
            agent = result["agent_used"]
            intent = result["intent"]
            confidence = result["intent_confidence"]
            answer = result["agent_response"]

            response = f"""**Agent:** {agent}
**Intent:** {intent} ({confidence:.2f})

{answer}"""

            if result.get("usage"):
                tokens = result["usage"].get("total_tokens", 0)
                response += f"\n\n---\n*Tokens: {tokens}*"

        if history is None:
            history = []

        history.append({"role": "user", "content": query})
        history.append({"role": "assistant", "content": response})

        if session_id in sessions:
            sessions[session_id] = history

        return "", history, session_id

    except Exception as e:
        error_msg = f"\u274c Error: {str(e)}"
        if history is None:
            history = []

        history.append({"role": "user", "content": query})
        history.append({"role": "assistant", "content": error_msg})

        if session_id in sessions:
            sessions[session_id] = history

        return "", history, session_id

def view_history(session_id):
    """View full session history from DB."""
    if not session_id:
        return "No active session"

    history = db_manager.load_session_history(session_id)
    if not history:
        return "No history found"

    output = f"## Session History: {session_id}\n\n"
    for msg in history:
        output += f"**You:** {msg['user_query']}\n\n"
        output += f"**AI ({msg['agent_used']}):** {msg['assistant_response']}\n\n"
        output += "---\n\n"

    return output

def create_session():
    """Create a new session and return session ID."""
    session_id = str(uuid.uuid4())
    sessions[session_id] = []
    return session_id

# Create Gradio interface
with gr.Blocks(title="Reverse Logistics Inspection Assistant", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # \U0001F50D Reverse Logistics Inspection Assistant
    Upload one or more component photos to run defect detection, then ask the assistant
    about specific results, batch-level QC stats, or repair guidance.
    """)

    session_state = gr.State(create_session)
    batch_id_state = gr.State(None)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 1. Upload & Inspect")
            file_upload = gr.File(
                label="Component images",
                file_count="multiple",
                file_types=["image"]
            )
            inspect_btn = gr.Button("Run Inspection", variant="primary")
            status_box = gr.Markdown()
            results_gallery = gr.Gallery(label="Detections", columns=3, height=300)
            results_table = gr.Dataframe(
                headers=["filename", "component_type", "categories_found", "decision", "action"],
                label="Batch Results"
            )

        with gr.Column(scale=1):
            gr.Markdown("### 2. Ask the Assistant")
            chatbot = gr.Chatbot(
                label="Conversation",
                height=400,
                type="messages"
            )
            msg = gr.Textbox(label="Your Question", placeholder="e.g., Why was item 2 rejected?")

            with gr.Row():
                submit = gr.Button("Send", variant="primary")
                clear = gr.Button("Clear Chat")

            gr.Markdown("### Sample Queries")
            sample_queries = gr.Dataset(
                components=[msg],
                samples=[
                    ["Give me a summary of this batch's results."],
                    ["Why was the first image flagged?"],
                    ["How should we repair components with corrosion?"],
                    ["What's the decision rule for Crack vs Wear?"]
                ],
                label="Click to try"
            )

            history_btn = gr.Button("View Full History")
            history_output = gr.Markdown()

    def respond(message, chat_history, session_id, batch_id):
        if not message:
            return "", chat_history, session_id
        if chat_history is None:
            chat_history = []
        new_msg, new_history, new_session = process_query(message, chat_history, session_id, batch_id)
        return new_msg, new_history, new_session

    def clear_chat_handler(session_id):
        if session_id in sessions:
            sessions[session_id] = []
        return [], session_id

    inspect_btn.click(
        run_inspection,
        [file_upload, batch_id_state],
        [results_gallery, results_table, batch_id_state, status_box]
    )

    submit.click(respond, [msg, chatbot, session_state, batch_id_state], [msg, chatbot, session_state])
    msg.submit(respond, [msg, chatbot, session_state, batch_id_state], [msg, chatbot, session_state])

    sample_queries.click(lambda x: x[0], [sample_queries], [msg])

    clear.click(clear_chat_handler, [session_state], [chatbot, msg])

    history_btn.click(view_history, [session_state], [history_output])

if __name__ == "__main__":
    demo.launch(
        share=False,
        server_name="127.0.0.1",
        server_port=7860
    )


In [ ]:
# 4. Initialize the database and RAG (FAISS) index
!python db.py
!python vision_tools.py
!python rag.py


In [ ]:
# 5. Launch the Gradio app
# share=True is required on Colab since 127.0.0.1 isn't reachable from your browser —
# Gradio will print a public *.gradio.live link instead.
from gradio_ui import demo

demo.launch(share=True)


## Notes

- **No bundled dataset:** detection uses Gemini's vision model zero-shot, since no labeled component image dataset or trained YOLOv8 model exists in this project. Bring your own photos of mechanical parts (or anything with visible wear/damage) to try it.
- **Going to production:** swap the body of `classify_component_image` in `vision_tools.py` for a real trained detector's inference call once you have a labeled dataset \u2014 `apply_decision_rule`, the agents, the graph, and the UI don't need to change.
- **Rate limits:** Gemini's free tier caps `gemini-2.5-flash` at 5 requests/minute, and each uploaded image costs one of those requests. Keep batches small (a handful of images) when testing.
- **Restarting:** if you restart the Colab runtime, re-run all cells from the top \u2014 the working directory, `.env`, database, and FAISS index are all wiped, so previous batches are lost.
- **Stopping the app:** Use Runtime \u2192 Interrupt execution to stop the Gradio server.
- **Updating this notebook:** since this app is embedded directly rather than imported from a separate module, any future change to the agent logic needs to be re-applied here too.
